# Track B: BRCA1 Mutation Walk -- Nucleotide Transformer Embeddings

**Purpose**: Embed the BRCA1 single-point mutation walk using the Nucleotide Transformer
(6-mer tokenized Transformer) and cache the results for cross-model analysis.

| Model | Architecture | Tokenization | Embedding |
|-------|-------------|-------------|-----------|
| **Nucleotide Transformer v2** (500M) | BERT Transformer | 6-mer (fixed k-mer) | Last hidden state mean-pool |


## Setup
1. Change Runtime to **GPU** (T4 sufficient for 500M model)
2. Run cells in order

Results are cached to `./results/interpolation_track_b/cache/` for the analysis notebook.

In [ ]:
# Cell: Install Dependencies
#
# NT v2 custom modeling code was written for transformers ~4.20-4.30.
# In transformers 5.x, several things were removed/moved. This cell
# restores everything so trust_remote_code works.
# (Adapted from NT_RealDNA_Experiment.ipynb)

print('Installing core dependencies...')
!pip install -q torch transformers matplotlib seaborn pandas scipy biopython numpy scikit-learn

import transformers, torch
print(f'transformers {transformers.__version__}')
print(f'torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# --- Patch transformers for Nucleotide Transformer compatibility ---
import torch.nn as nn
import transformers.pytorch_utils as _ptu
from transformers import PretrainedConfig

# Patch 1: Restore missing functions in pytorch_utils
if not hasattr(_ptu, 'find_pruneable_heads_and_indices'):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index
    _ptu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices
    print('Patched: find_pruneable_heads_and_indices')

if not hasattr(_ptu, 'prune_linear_layer'):
    def prune_linear_layer(layer, index, dim=0):
        index = index.to(layer.weight.device)
        W = layer.weight.index_select(dim, index).clone().detach()
        if layer.bias is not None:
            b = layer.bias.clone().detach() if dim == 1 else layer.bias[index].clone().detach()
        new_size = list(layer.weight.size())
        new_size[dim] = len(index)
        new_layer = nn.Linear(new_size[1], new_size[0], bias=layer.bias is not None).to(layer.weight.device)
        new_layer.weight.requires_grad = False
        new_layer.weight.copy_(W.contiguous())
        new_layer.weight.requires_grad = True
        if layer.bias is not None:
            new_layer.bias.requires_grad = False
            new_layer.bias.copy_(b.contiguous())
            new_layer.bias.requires_grad = True
        return new_layer
    _ptu.prune_linear_layer = prune_linear_layer
    print('Patched: prune_linear_layer')

# Patch 2: Restore missing PreTrainedModel methods removed in transformers 5.x
from transformers import PreTrainedModel as _PTM

if not hasattr(_PTM, 'get_head_mask'):
    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            if head_mask.dim() == 1:
                head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
                head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
            elif head_mask.dim() == 2:
                head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
            head_mask = head_mask.to(dtype=next(self.parameters()).dtype)
            if is_attention_chunked:
                head_mask = head_mask.unsqueeze(-1)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask
    _PTM.get_head_mask = get_head_mask
    print('Patched: get_head_mask')

if not hasattr(_PTM, 'get_extended_attention_mask'):
    def get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        if attention_mask.dim() == 3:
            extended = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended = attention_mask[:, None, None, :]
        else:
            raise ValueError(f'Wrong shape for attention_mask: {attention_mask.shape}')
        if dtype is None:
            dtype = next(self.parameters()).dtype
        extended = extended.to(dtype=dtype)
        extended = (1.0 - extended) * torch.finfo(dtype).min
        return extended
    _PTM.get_extended_attention_mask = get_extended_attention_mask
    print('Patched: get_extended_attention_mask')

if not hasattr(_PTM, 'invert_attention_mask'):
    def invert_attention_mask(self, encoder_attention_mask):
        if encoder_attention_mask.dim() == 3:
            extended = encoder_attention_mask[:, None, :, :]
        elif encoder_attention_mask.dim() == 2:
            extended = encoder_attention_mask[:, None, None, :]
        else:
            raise ValueError(f'Wrong shape: {encoder_attention_mask.shape}')
        dtype = next(self.parameters()).dtype
        extended = extended.to(dtype=dtype)
        extended = (1.0 - extended) * torch.finfo(dtype).min
        return extended
    _PTM.invert_attention_mask = invert_attention_mask
    print('Patched: invert_attention_mask')

# Patch 3: Fix all_tied_weights_keys / tie_weights compatibility
import transformers.modeling_utils as _mu
if not getattr(_mu, '_nt_patched', False):
    orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized
    def safe_mark(self):
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        return orig_mark(self)
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = safe_mark

    orig_finalize = _mu.PreTrainedModel._finalize_load_state_dict
    @staticmethod
    def safe_finalize(model, load_config, load_info):
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        orig_tie = model.tie_weights
        def _patched_tie(missing_keys=None, recompute_mapping=False, **kwargs):
            return orig_tie()
        model.tie_weights = _patched_tie
        return orig_finalize(model, load_config, load_info)
    _mu.PreTrainedModel._finalize_load_state_dict = safe_finalize
    _mu._nt_patched = True
    print('Patched: all_tied_weights_keys / tie_weights')

# Patch 4: Restore missing default config attributes
_config_defaults = {
    'is_decoder': False,
    'add_cross_attention': False,
    'chunk_size_feed_forward': 0,
    'is_encoder_decoder': False,
    'tie_word_embeddings': True,
}
for attr, default in _config_defaults.items():
    if not hasattr(PretrainedConfig, attr):
        setattr(PretrainedConfig, attr, default)
        print(f'Patched config default: {attr}={default}')

print('\nDone!')

In [ ]:
# Cell: Configuration

import os
import sys
import numpy as np
import torch

sys.path.insert(0, '.')

# --- Output paths ---
OUTPUT_BASE = './results/interpolation_track_b/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# --- BRCA1 ---
BRCA1_REGION_LEN = 2000   # 2kb core region for mutation walk
BRCA1_FLANKING   = 16_384 # Total region to fetch (with flanking context)
N_EXTRA_SNPS     = 120
SEED             = 320

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Output: {OUTPUT_BASE}')

# --- Nucleotide Transformer ---
# Using the v2 500M multi-species model (good balance of size and quality)
NT_MODEL = 'InstaDeepAI/nucleotide-transformer-v2-500m-multi-species'
NT_MAX_LENGTH = 1000  # in tokens (each token = 6-mer, so ~6000 bp)
print(f'Nucleotide Transformer model: {NT_MODEL}')
print(f'Max tokens: {NT_MAX_LENGTH} (~{NT_MAX_LENGTH * 6:,} bp)')

In [ ]:
# Cell: Download BRCA1 Region & Define Mutation Walk Endpoints

import urllib.request
import json as _json


def fetch_brca1_region(total_len=BRCA1_FLANKING, core_len=BRCA1_REGION_LEN):
    '''Fetch BRCA1 region from UCSC (GRCh38/hg38).

    Downloads total_len bp centered on the C61G variant position.
    The central core_len bp is the "mutation zone" where SNPs are introduced.
    '''
    center = 43_104_121  # BRCA1 C61G position (GRCh38)
    start = center - total_len // 2
    end = start + total_len
    core_start = (total_len - core_len) // 2
    core_end = core_start + core_len

    url = (f'https://api.genome.ucsc.edu/getData/sequence'
           f'?genome=hg38;chrom=chr17;start={start};end={end}')

    print(f'Fetching BRCA1 region chr17:{start:,}-{end:,} ({total_len:,} bp)...')
    try:
        req = urllib.request.Request(url)
        req.add_header('User-Agent', 'GeometricTax-Experiment/1.0')
        with urllib.request.urlopen(req, timeout=30) as response:
            data = _json.loads(response.read().decode())
            seq = data.get('dna', '').upper()
            if len(seq) >= total_len * 0.9:
                seq = seq[:total_len]
                rng = np.random.default_rng(SEED)
                seq = ''.join(
                    c if c in 'ACGT' else rng.choice(list('ACGT'))
                    for c in seq
                )
                print(f'  Downloaded {len(seq)} bp from UCSC')
                print(f'  Core region: positions [{core_start}:{core_end}] ({core_len} bp)')
                return seq, core_start, core_end
    except Exception as e:
        print(f'  UCSC download failed: {e}')

    # Fallback: synthetic
    print('  Generating synthetic BRCA1-like sequence as fallback...')
    rng = np.random.default_rng(SEED + 17)
    bases = rng.choice(list('ACGT'), size=total_len, p=[0.29, 0.21, 0.21, 0.29])
    seq = ''.join(bases)
    print(f'  Generated {len(seq)} bp synthetic BRCA1-like sequence')
    return seq, core_start, core_end


def create_pathogenic_variant(full_seq, core_start, core_end,
                              n_extra_snps=N_EXTRA_SNPS, seed=SEED):
    '''Create a pathogenic variant by mutating only the core region.'''
    rng = np.random.default_rng(seed)
    bases = list(full_seq)
    core_center = (core_start + core_end) // 2
    mutated_positions = set()

    # 1. Pathogenic mutation at core center
    alt_base = 'G' if bases[core_center] != 'G' else 'C'
    print(f'  Pathogenic mutation at position {core_center}: {bases[core_center]} -> {alt_base}')
    bases[core_center] = alt_base
    mutated_positions.add(core_center)

    # 2. Additional random SNPs within core region only
    available = [i for i in range(core_start, core_end) if i not in mutated_positions]
    snp_positions = rng.choice(available, size=min(n_extra_snps, len(available)),
                                replace=False)
    snp_positions.sort()

    for pos in snp_positions:
        original = bases[pos]
        alternatives = [b for b in 'ACGT' if b != original]
        bases[pos] = rng.choice(alternatives)
        mutated_positions.add(pos)

    mutant = ''.join(bases)
    hamming = sum(a != b for a, b in zip(full_seq, mutant))
    print(f'  Total mutations: {hamming} (all within core [{core_start}:{core_end}])')
    return mutant, sorted(mutated_positions)


# Generate endpoints
full_wt_seq, CORE_START, CORE_END = fetch_brca1_region()
full_mut_seq, mutation_positions = create_pathogenic_variant(
    full_wt_seq, CORE_START, CORE_END)

print(f'\nFull sequence length: {len(full_wt_seq)}')
print(f'Core region: [{CORE_START}:{CORE_END}] ({CORE_END - CORE_START} bp)')
print(f'Hamming distance: {sum(a != b for a, b in zip(full_wt_seq, full_mut_seq))}')

wildtype_seq = full_wt_seq
mutant_seq = full_mut_seq

In [ ]:
# Cell: Generate Single-Point Mutation Walk

def single_point_mutation_walk(seq_start, seq_end, seed=SEED):
    """Create a mutation walk from seq_start to seq_end, one base at a time."""
    assert len(seq_start) == len(seq_end), 'Sequences must be same length'

    diff_positions = [i for i in range(len(seq_start))
                      if seq_start[i] != seq_end[i]]

    rng = np.random.default_rng(seed)
    rng.shuffle(diff_positions)

    current = list(seq_start)
    walk = [''.join(current)]
    step_positions = []

    for pos in diff_positions:
        current[pos] = seq_end[pos]
        walk.append(''.join(current))
        step_positions.append(pos)

    assert walk[-1] == seq_end, 'Walk did not reach target'
    return walk, step_positions


mutation_walk, walk_positions = single_point_mutation_walk(wildtype_seq, mutant_seq)

n_steps = len(mutation_walk)
print(f'Mutation walk: {n_steps} steps (start + {n_steps - 1} mutations)')

center_pos = (CORE_START + CORE_END) // 2
pathogenic_step = None
for i, pos in enumerate(walk_positions):
    if pos == center_pos:
        pathogenic_step = i + 1
        break
print(f'Pathogenic C61G mutation occurs at step {pathogenic_step}/{n_steps - 1}')

walk_cache = f'{CACHE_DIR}/brca1_mutation_walk.npz'
np.savez(walk_cache, walk=mutation_walk, positions=walk_positions,
         pathogenic_step=pathogenic_step)
print(f'Walk cached to {walk_cache}')

In [ ]:
# Cell: Load Nucleotide Transformer
#
# CRITICAL: Must use AutoModelForMaskedLM, not AutoModel.
# The checkpoint includes lm_head.* weights. Using AutoModel causes
# size mismatches (intermediate dense 8192 vs 4096) because the base
# EsmModel has different architecture assumptions than EsmForMaskedLM.
#
# Embeddings: last hidden state via output_hidden_states=True, mean-pooled.

import torch
import gc
from transformers import AutoTokenizer, AutoModelForMaskedLM


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def load_nucleotide_transformer(model_name=NT_MODEL, batch_size=4):
    """Load Nucleotide Transformer and return embedding function.

    Uses AutoModelForMaskedLM (not AutoModel) because the checkpoint
    includes the LM head weights. We extract the last hidden state
    and mean-pool over non-padding positions.
    """
    print(f'Loading {model_name}...')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForMaskedLM.from_pretrained(
        model_name, trust_remote_code=True,
    ).to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6

    max_length = tokenizer.model_max_length
    if max_length > 100000:
        max_length = 1024
    print(f'  Params: {n_params:.1f}M')
    print(f'  Vocab size: {tokenizer.vocab_size}')
    print(f'  Max token length: {max_length}')

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size
        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1
            if batch_idx % 10 == 0 or batch_idx == n_batches:
                print(f'  Batch {batch_idx}/{n_batches}', end='\r')

            tokens = tokenizer(
                batch, return_tensors='pt', padding=True,
                truncation=True, max_length=max_length,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}

            outputs = model(**tokens, output_hidden_states=True)
            # Use last hidden state from hidden_states tuple
            hidden = outputs.hidden_states[-1]

            # Mean-pool over non-padding positions
            attention_mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)

            embeddings.append(pooled.cpu().numpy())
        print()
        return np.concatenate(embeddings, axis=0).astype(np.float32)

    print(f'Nucleotide Transformer ready: {n_params:.1f}M params')
    return embed, model, tokenizer, n_params


nt_embed, nt_model, nt_tokenizer, nt_params = load_nucleotide_transformer()

In [ ]:
# Cell: Embed Mutation Walk with Nucleotide Transformer
#
# NT uses 6-mer tokenization. With NT_MAX_LENGTH=1000 tokens,
# we can process ~6000 bp. We use the full 16kb sequence but it will
# be truncated by the tokenizer. For consistency with DNABERT-2,
# we also try the core region.

nt_cache = f'{CACHE_DIR}/nucleotide_transformer_brca1_walk.npy'

if os.path.exists(nt_cache):
    print('Loading cached Nucleotide Transformer embeddings...')
    nt_embeddings = np.load(nt_cache)
else:
    # NT can handle longer sequences than DNABERT-2.
    # Use a wider window centered on the core region to give context,
    # but still within the tokenizer's limit.
    # ~6000 bp fits in 1000 6-mer tokens, so use 6kb centered on core.
    context_bp = NT_MAX_LENGTH * 6  # ~6000 bp
    pad = (context_bp - (CORE_END - CORE_START)) // 2
    window_start = max(0, CORE_START - pad)
    window_end = min(len(mutation_walk[0]), window_start + context_bp)

    nt_walk = [seq[window_start:window_end] for seq in mutation_walk]
    print(f'Embedding {len(nt_walk)} walk steps with Nucleotide Transformer...')
    print(f'(Using window [{window_start}:{window_end}], {window_end - window_start} bp)')

    # Check tokenization length
    test_tokens = nt_tokenizer(nt_walk[0], return_tensors='pt')
    n_tok = test_tokens['input_ids'].shape[1]
    print(f'Window tokenizes to ~{n_tok} 6-mer tokens')

    nt_embeddings = nt_embed(nt_walk)
    np.save(nt_cache, nt_embeddings)
    print(f'Cached to {nt_cache}')

print(f'Nucleotide Transformer embeddings: {nt_embeddings.shape}')

# Free GPU memory
del nt_model
cleanup_gpu()
print('Nucleotide Transformer model freed from GPU')

In [ ]:
# Cell: Quick Sanity Check

from scipy.spatial.distance import cosine

print('Sanity check: cosine distance from wildtype')
for step_idx in [0, 10, 50, len(nt_embeddings) - 1]:
    if step_idx < len(nt_embeddings):
        d = cosine(nt_embeddings[0], nt_embeddings[step_idx])
        print(f'  Step {step_idx:4d}: cosine dist = {d:.6f}')

# Consecutive step distances
dists = [cosine(nt_embeddings[i], nt_embeddings[i + 1])
         for i in range(len(nt_embeddings) - 1)]
dists = np.array(dists)
print(f'\nConsecutive step cosine distances:')
print(f'  mean={dists.mean():.6f}, std={dists.std():.6f}, max={dists.max():.6f}')
print(f'  smoothness (mean/max): {dists.mean() / (dists.max() + 1e-8):.4f}')
print(f'\nNucleotide Transformer embeddings cached and ready for analysis notebook.')